# Machine Learning with Spark MLlib: Emotion Classification Baseline

**Consulting Engagement: BigDataCompany - Big Data Analytics Architecture**  
*Course: Big Data Analysis (BSc Labs) - NOVA IMS*  
*Client Prototype: NLP Machine Learning Classification Pipeline*

---

### Business Objective
To support mental health professionals and social media moderation platforms, our consulting team is developing an end-to-end, distributed **Natural Language Processing (NLP) classification engine**. The objective is to automatically classify social media texts into one of six emotional states:
`Positive`, `Neutral`, `Anger`, `Depression`, `Sadness`, or `Anxiety`.

### Architecture: Spark MLlib Unified Pipeline
In distributed machine learning, constructing an end-to-end **`Pipeline`** is essential:
1. **Encapsulation:** Feature transformations and model estimation are packaged into a single reusable object.
2. **Leakage Prevention:** Transformers (such as IDF) are fitted solely on training splits, eliminating test-set leakage.
3. **Deployment Readiness:** The resulting `PipelineModel` can be serialized and deployed directly into batch or streaming inference environments (such as our upcoming **Spark Streaming** bonus module).

```
Cleaned Text ──▶ [Tokenizer] ──▶ [StopWordsRemover] ──▶ [HashingTF] ──▶ [IDF] ──▶ [LogisticRegression] ──▶ Predictions
```


In [ ]:
import os
import sys

from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField, StringType, LongType, 
    DoubleType, BooleanType, TimestampType
)
from pyspark.sql.functions import col, count, desc, when, length, size, split

from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.feature import (
    Tokenizer, StopWordsRemover, HashingTF, IDF, 
    StringIndexer, IndexToString
)
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Initialize Spark Session with MLlib tuning
spark = SparkSession.builder \
    .appName("MLlib_Emotion_Classification_Baseline") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

# Path resolver for Databricks DBFS and Local Repositories
def resolve_lakehouse_path(subpath):
    candidates = [
        f"/FileStore/lakehouse/{subpath}",
        f"dbfs:/FileStore/lakehouse/{subpath}",
        f"../data/lakehouse/{subpath}",
        f"data/lakehouse/{subpath}"
    ]
    for p in candidates:
        clean_p = p.replace("dbfs:", "")
        if os.path.exists(clean_p) or p.startswith("/FileStore") or p.startswith("dbfs:"):
            return p
    return f"../data/lakehouse/{subpath}"

silver_text_path = resolve_lakehouse_path("silver/mental_health")
print(f"Target Silver Lakehouse Path: {silver_text_path}")


## 1. Silver Lakehouse Ingestion & Verification

We ingest the pre-cleansed Silver table (`silver/mental_health`) generated in the Preprocessing Lakehouse pipeline. This table contains verified, non-empty text records alongside numerical label indices ($0.0$ to $5.0$) fitted via `StringIndexer`.


In [ ]:
# Fallback ingestion helper if running before preprocessing persistence
try:
    # Try reading Delta format
    silver_df = spark.read.format("delta").load(silver_text_path)
    print("✅ Ingested Silver table from Delta Lake format.")
except Exception as e_delta:
    try:
        # Try reading Parquet format
        silver_df = spark.read.format("parquet").load(silver_text_path)
        print("✅ Ingested Silver table from Parquet format.")
    except Exception as e_parquet:
        print("⚠️ Silver table not found on disk. Loading raw CSV with inline fallback cleaning...")
        raw_csv_path = "../Datasets/mental_health.csv" if os.path.exists("../Datasets/mental_health.csv") else "Datasets/mental_health.csv"
        raw_df = spark.read.csv(raw_csv_path, header=True, inferSchema=True, multiLine=True, escape='"')
        
        from pyspark.sql.functions import regexp_replace, lower, trim
        clean_df = raw_df.filter(col("Text").isNotNull() & (length(trim(col("Text"))) > 2)) \
            .withColumn("Cleaned_Text", regexp_replace(col("Text"), r"http\S+|www\.\S+", "")) \
            .withColumn("Cleaned_Text", regexp_replace(col("Cleaned_Text"), r"@\w+", "")) \
            .withColumn("Cleaned_Text", regexp_replace(col("Cleaned_Text"), r"[^a-zA-Z\s]", " ")) \
            .withColumn("Cleaned_Text", lower(col("Cleaned_Text"))) \
            .withColumn("Cleaned_Text", trim(regexp_replace(col("Cleaned_Text"), r"\s+", " "))) \
            .filter(length(col("Cleaned_Text")) > 2)
            
        indexer = StringIndexer(inputCol="Emotion", outputCol="label")
        silver_df = indexer.fit(clean_df).transform(clean_df)

# Verify schema and record counts
total_records = silver_df.count()
print(f"Total ML-Ready Records: {total_records:,}")
silver_df.select("ID", "Cleaned_Text", "Emotion", "label").show(5, truncate=60)


## 2. Distributed Train / Test Splitting

To ensure statistical rigor and reproducibility, we partition the dataset into:
* **Training Set (80%):** Used for fitting token weights, inverse document frequencies, and classifier coefficients.
* **Test Set (20%):** Held-out partition strictly reserved for unbiased performance evaluation.

We set a deterministic seed (`seed=42`) across the distributed Spark workers.


In [ ]:
# Deterministic 80/20 train/test split
train_df, test_df = silver_df.randomSplit([0.8, 0.2], seed=42)

# Cache training and testing partitions in cluster memory for high iteration speed
train_df.cache()
test_df.cache()

train_count = train_df.count()
test_count = test_df.count()

print("=== Distributed Dataset Split ===")
print(f"Training Partition: {train_count:,} records ({train_count/total_records*100:.1f}%)")
print(f"Testing Partition:  {test_count:,} records ({test_count/total_records*100:.1f}%)")


## 3. End-to-End Spark MLlib Feature Pipeline

We construct a modular Spark ML pipeline consisting of 5 stages:
1. **`Tokenizer`:** Segments continuous cleaned text into discrete token arrays.
2. **`StopWordsRemover`:** Filters common English stopwords that do not convey emotional sentiment (e.g., *the, is, at, which*).
3. **`HashingTF`:** Employs the distributed MurmurHash3 algorithm to map tokens to feature indices with a fixed vocabulary dimension ($N = 10,000$). This avoids maintaining an expensive global dictionary across worker nodes.
4. **`IDF`:** Inverse Document Frequency rescales feature weights, amplifying rare, emotionally expressive terms while downweighting ubiquitous words.
5. **`LogisticRegression`:** Multi-class multinomial logistic regression classifier modeling the posterior class probabilities:
   $$P(Y = k \mid \mathbf{x}) = \frac{e^{\mathbf{w}_k^T \mathbf{x}}}{\sum_{j=1}^K e^{\mathbf{w}_j^T \mathbf{x}}}$$


In [ ]:
# Stage 1: Tokenizer
tokenizer = Tokenizer(inputCol="Cleaned_Text", outputCol="words")

# Stage 2: StopWordsRemover
stopwords_remover = StopWordsRemover(inputCol="words", outputCol="filtered_words")

# Stage 3: HashingTF (10,000 features for optimal balance of expressiveness and memory efficiency)
hashing_tf = HashingTF(inputCol="filtered_words", outputCol="raw_features", numFeatures=10000)

# Stage 4: IDF
idf = IDF(inputCol="raw_features", outputCol="features")

# Stage 5: Multinomial Logistic Regression Classifier
lr = LogisticRegression(
    featuresCol="features", 
    labelCol="label", 
    maxIter=20, 
    regParam=0.01, 
    elasticNetParam=0.0,
    family="multinomial"
)

# Assemble end-to-end Pipeline
pipeline = Pipeline(stages=[
    tokenizer, 
    stopwords_remover, 
    hashing_tf, 
    idf, 
    lr
])

print("✅ Pipeline stages assembled successfully.")
print("Fitting end-to-end PipelineModel on training data...")
pipeline_model = pipeline.fit(train_df)
print("✅ Model training complete.")


## 4. Multi-Class Model Evaluation

We apply the fitted `PipelineModel` to the held-out test set and evaluate performance across multiple standard metrics using `MulticlassClassificationEvaluator`.

Given the slight class imbalance identified during EDA (e.g. `Anxiety` at ~11% vs `Positive` at ~23%), we prioritize **Weighted F1-Score** over raw Accuracy.


In [ ]:
# Transform held-out test data
predictions_df = pipeline_model.transform(test_df)

# Setup Multi-class Evaluators
evaluator_acc = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
evaluator_f1 = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedFMeasure")
evaluator_prec = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedPrecision")
evaluator_rec = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedRecall")

accuracy = evaluator_acc.evaluate(predictions_df)
weighted_f1 = evaluator_f1.evaluate(predictions_df)
weighted_prec = evaluator_prec.evaluate(predictions_df)
weighted_rec = evaluator_rec.evaluate(predictions_df)

print("=" * 50)
print("   SPARK MLLIB BASELINE PERFORMANCE (TEST SET)   ")
print("=" * 50)
print(f"Overall Accuracy:        {accuracy * 100:.2f}%")
print(f"Weighted F1-Score:       {weighted_f1 * 100:.2f}%")
print(f"Weighted Precision:      {weighted_prec * 100:.2f}%")
print(f"Weighted Recall:         {weighted_rec * 100:.2f}%")
print("=" * 50)


In [ ]:
# Sample test predictions with original text snippet
predictions_df.select(
    "Cleaned_Text", 
    "Emotion", 
    "label", 
    "prediction"
).show(10, truncate=75)


In [ ]:
# Confusion Matrix: Actual vs Predicted Labels
print("=== Confusion Matrix Summary (Actual Label vs Predicted Label) ===")
confusion_matrix = predictions_df.groupBy("label") \
    .pivot("prediction", [0.0, 1.0, 2.0, 3.0, 4.0, 5.0]) \
    .count() \
    .na.fill(0) \
    .orderBy("label")

confusion_matrix.show()


## 5. Model Serialization & Handoff to Streaming (Bonus)

A major advantage of Spark's `PipelineModel` architecture is full pipeline portability. We persist the trained pipeline model to disk/DBFS. 

This saved artifact will be loaded directly into **Notebook 5 (`GX_Streaming.ipynb`)**, allowing real-time streaming data to be tokenized, converted to TF-IDF vectors, and classified without rewriting any feature engineering logic.


In [ ]:
# Target Model Path for DBFS or Local Repository
model_save_path = "/FileStore/models/mllib_emotion_baseline" if os.path.exists("/FileStore") else "../models/mllib_emotion_baseline"
os.makedirs(os.path.dirname(model_save_path), exist_ok=True)

try:
    pipeline_model.write().overwrite().save(model_save_path)
    print(f"✅ PipelineModel serialized successfully to: {model_save_path}")
except Exception as e:
    print(f"Note on model serialization: {e}")


## 6. Executive Consulting Takeaways & Next Milestones

### Key Insights from Baseline Deployment:
1. **Strong Baseline Efficiency:** Multinomial Logistic Regression with HashingTF-IDF yields strong classification accuracy and weighted F1 on the 6-class problem.
2. **Pipeline Reusability:** Encapsulating tokenization, stop-words removal, feature hashing, and IDF weighting within a single Spark `Pipeline` ensures zero training-test data leakage and reproducible deployment.
3. **Foundation for Next Iteration (Milestone 3.2):**
   * Compare against alternative classifiers: **Naive Bayes** (ideal for multinomial document frequencies) and **Linear Support Vector Machines (LinearSVC / OneVsRest)**.
   * Implement distributed Hyperparameter Optimization via `ParamGridBuilder` and `CrossValidator` across regularization parameters ($L_1 / L_2$ elastic net mixing) and HashingTF dimensions.
   * Export optimal pipeline for live scoring in **`GX_Streaming.ipynb`**.
